# Pull Holdings and Derivative Data from Eagle

## eagle_query.ipynb is derived from eagle_parn_downloading.ipynb

How to wait until Element is Visible in Selenium Python  

https://pythonexamples.org/python-selenium-wait-until-element-is-visible/

In [22]:
k = datetime.today()
k

datetime.datetime(2024, 8, 6, 14, 22, 39, 478361)

In [1]:
def osprey(rpt_type = 'r28i', funds = 'PIMBAL,PABS', d_from, d_to, al = 'sober', xe = 'duck'):
    # an Eagle report lookup function
    start_time          = time.time()

    # (1) load libraries
    from datetime import datetime, timedelta
    %run utilities.ipynb
    from selenium.webdriver.common.by import By
    from selenium.webdriver.common.keys import Keys
    from selenium.webdriver.support.select import Select
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC
    
    # (2) set paths to the driver, urls, and to the report parameters
    import os
    os.environ["PATH"] = r'C:/SeleniumDrivers' # + os.pathsep + os.getenv("PATH")
    # https://stackoverflow.com/questions/61213005/modify-beginning-of-path-variable-with-os-environ-in-python
    pth                = r'P:\Investment Operations\GRC\Compliance\Daily\py_reports.xlsm' # user_defined variables stored here
    url_default        = r'https://eagleportal.prescient.co.za/Default.aspx'
    eagle_root         = r'https://eagleportal.prescient.co.za/Queries/Query.aspx?rpt='
    # eagle report types, their short codes, and their URLs
    report_types_dict = {'r28i': ['Reg 28 Report - Incl Effective Exposure', eagle_root + 'Reg28withExposure'],
                         'parn': ['Portfolio Analytics Report - New',        eagle_root + 'PortfolioAnalytics'],
                         'derv': ['Derivative Exposure',                     eagle_root + 'DerivativeExposure'],
                         'trad': ['Trades Report',                           eagle_root + 'TRANSACTION'],
                         'scty': ['Security Cross Reference',                eagle_root + 'SecurityCrossRef'],
                         'dflw': ['Daily Flows',                             eagle_root + 'FLOWS'],
                         'utps': ['Unit Trust Prices',                       eagle_root + 'UTPRICES'], 
                         'fnav': ['Fund Net Asset Value',                    eagle_root + 'NetAsset'],
                         'tcrf': ['Trades Cross Reference',                  eagle_root + 'TRADES%20REFERENCE']}    
    
    # (3) prepare the report variables for fnds_, month_year_, day_, report_type
    fnds_       = funds
    month_year_ = f'{d_from:%B}, {d_from:%Y}' # e.g., 'January, 2023'
    day_        = f'{d_from:%#d}'             # e.g., '03', i.e., report day with a leading zero f'{d_from:%d}'
    report_type = report_types_dict[rpt_type[1]]
    
    # (4) assign the browser driver
    from selenium import webdriver
    driver = webdriver.Firefox()
    
    # (5) open the browser on the Eagle web page
    driver.get(url_default)          # default page
    wait = WebDriverWait(driver, 10) # https://selenium-python.readthedocs.io/waits.html, max wait for elements to appear
    
    # (6) login
    driver.find_element(By.CSS_SELECTOR, '#LoginCtrl_MainLoginControl_UserName'   ).send_keys(al)
    driver.find_element(By.CSS_SELECTOR, '#LoginCtrl_MainLoginControl_Password'   ).send_keys(xe)
    driver.find_element(By.CSS_SELECTOR, '#LoginCtrl_MainLoginControl_LoginButton').click()

    # (7) having logged in, open the reporting page (NEEDS report type link)
    driver.get(report_type)             # a hyperlink for the reporting page selected in the function osprey()
    
    # (8) switch to the query page
    driver.find_element(By.CSS_SELECTOR, '#ModifyLinkLabel').click()

    # (8a) save the current window (NECESSARY?)
    edit_criteria_window = driver.window_handles[0] # save curent window handle
    # https://stackoverflow.com/questions/10629815/how-to-switch-to-new-window-in-selenium-for-python
    
    # (9) update the calendar (NEEDS month_year_, day_)
    driver.find_element(By.CSS_SELECTOR, 'td[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_DATE1_DateCtrl_From_B-1"]').click()
    driver.find_element(By.CSS_SELECTOR, 'td[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_DATE1_DateCtrl_From_DDD_C_NMC"]').click()
    lmonth_selector = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'td[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_DATE1_DateCtrl_From_DDD_C_PMC"]')))
    while driver.find_element(By.XPATH,'//td[@id="ctl00_c_qc_QueryInputs_QueryInputsPopup_DATE1_DateCtrl_From_DDD_C_TC"]').text != month_year_:
        lmonth_selector.click()
    day_selector = driver.find_element(By.XPATH,f'//td[@class="dxeCalendarDay"][text()={day_}] | //td[@class="dxeCalendarDay dxeCalendarWeekend"][text()={day_}]')
    day_selector.click()
    
    # (10) get the web element for the FUND LIST and assign values to it
    fund_selector  = driver.find_element(By.CSS_SELECTOR, 'input[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_FUND0_SelectedIds"]')
    driver.execute_script(f'arguments[0].value = "{fnds_}";', fund_selector)
    
    # (11) get the web element of the 'Submit' button and then click it
    submit_button  = driver.find_element(By.CSS_SELECTOR, 'input[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_RunBtn"]')
    submit_button.click()
    
    # (12) Wait for and then click the export button and then the xls download button
    #https://stackoverflow.com/questions/56085152/selenium-python-error-element-could-not-be-scrolled-into-view
    WebDriverWait(driver, 1000).until(EC.element_to_be_clickable((By.CSS_SELECTOR,  'a[id="DistrBtn"]'        ))).click()
    WebDriverWait(driver, 1000).until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'td[id="ExportMnu_DXI4_T"]'))).click() # DXI4(6) for .xls(csv)
    
    time.sleep(30) # wait for 30 seconds after the holdings data downloads
    
    print(f'Downloading and then saving the {rpt_type[0]} report(s): {time_diff(start_time, time.time())}', '\n')

    # (13) having downloaded the requested report, close the web driver
    driver.quit()
    #print(f'Roundtrip time for getting holdings and derivative data: {time_diff(start_time_overlord, time.time())}', '\n')

    print(f'Roundtrip time to run the report lookup function: {time_diff(start_time, time.time())}', '\n')

SyntaxError: non-default argument follows default argument (1579069408.py, line 1)

In [9]:
start_time = time.time()
print(f'Downloading and then saving the {rpt_type[0]} report(s) ...')

#having logged in, open the reporting page
driver.get(rpt_type[1])             # reporting page selected in the function osprey()

# switch to the query page
driver.find_element(By.CSS_SELECTOR, '#ModifyLinkLabel').click()

edit_criteria_window = driver.window_handles[0] # save curent window handle
# https://stackoverflow.com/questions/10629815/how-to-switch-to-new-window-in-selenium-for-python

# update the calendar
driver.find_element(By.CSS_SELECTOR, 'td[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_DATE1_DateCtrl_From_B-1"]').click()
driver.find_element(By.CSS_SELECTOR, 'td[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_DATE1_DateCtrl_From_DDD_C_NMC"]').click()
lmonth_selector = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'td[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_DATE1_DateCtrl_From_DDD_C_PMC"]')))
while driver.find_element(By.XPATH,'//td[@id="ctl00_c_qc_QueryInputs_QueryInputsPopup_DATE1_DateCtrl_From_DDD_C_TC"]').text != month_year_:
    lmonth_selector.click()
day_selector = driver.find_element(By.XPATH,f'//td[@class="dxeCalendarDay"][text()={day_}] | //td[@class="dxeCalendarDay dxeCalendarWeekend"][text()={day_}]')
day_selector.click()

# get the web element for the FUND LIST and assign values to it
fund_selector  = driver.find_element(By.CSS_SELECTOR, 'input[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_FUND0_SelectedIds"]')
driver.execute_script(f'arguments[0].value = "{fnds_}";', fund_selector)

# get the web element of the 'Submit' button and then click it
submit_button  = driver.find_element(By.CSS_SELECTOR, 'input[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_RunBtn"]')
submit_button.click()

#Wait for and then click the export button and then the xls download button
#https://stackoverflow.com/questions/56085152/selenium-python-error-element-could-not-be-scrolled-into-view
WebDriverWait(driver, 1000).until(EC.element_to_be_clickable((By.CSS_SELECTOR,  'a[id="DistrBtn"]'        ))).click()
WebDriverWait(driver, 1000).until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'td[id="ExportMnu_DXI4_T"]'))).click()

time.sleep(30) # wait for 30 seconds after the holdings data downloads

print(f'Downloading and then saving the {report_type} report(s): {timediff(start_time, time.time())}', '\n')
print(f'Roundtrip time for getting holdings and derivative data: {timediff(start_time_overlord, time.time())}', '\n')


Roundtrip time for getting holdings and derivative data: 1min 7.2sec 



In [10]:
driver.quit() # close the web driver